# Neural Network SMS Text Classifier

Build a neural-network model that classifies SMS messages as **ham** (normal) or **spam**.

The notebook follows the freeCodeCamp challenge structure: load the supplied training and testing data, vectorize the message text, train a neural network, and implement `predict_message(message)` returning `[spam_probability, label]`.

Run the original freeCodeCamp final test cell to verify the challenge.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix


## 2. Load the SMS Spam Collection data

In [ ]:
# Download the SMS Spam Collection dataset.
# The UCI file is tab-separated: label<TAB>message.
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(DATA_URL, sep="\t", header=None, names=["label", "message"])
df.head()


## 3. Prepare training and testing data

In [ ]:
# Convert labels to numeric values: ham=0, spam=1.
label_encoder = LabelEncoder()
df["label_num"] = label_encoder.fit_transform(df["label"])

# Stratified split so both sets preserve the ham/spam proportions.
from sklearn.model_selection import train_test_split

train_messages, test_messages, train_labels, test_labels = train_test_split(
    df["message"].astype(str),
    df["label_num"].astype("float32"),
    test_size=0.20,
    random_state=42,
    stratify=df["label_num"]
)

print("Training samples:", len(train_messages))
print("Testing samples:", len(test_messages))
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


## 4. Convert SMS text into numeric features

In [ ]:
# TF-IDF converts each message into a numeric vector.
# Fit only on training messages to avoid data leakage.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=12000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_messages).astype("float32").toarray()
X_test = vectorizer.transform(test_messages).astype("float32").toarray()

y_train = np.asarray(train_labels, dtype="float32")
y_test = np.asarray(test_labels, dtype="float32")

print("Training matrix:", X_train.shape)
print("Testing matrix:", X_test.shape)


## 5. Build and train the neural network

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.15,
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


## 6. Evaluate the model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")

predicted_probabilities = model.predict(X_test, verbose=0).ravel()
predicted_labels = (predicted_probabilities >= 0.5).astype(int)

print(classification_report(y_test.astype(int), predicted_labels, target_names=["ham", "spam"]))
print("Confusion matrix:")
print(confusion_matrix(y_test.astype(int), predicted_labels))


## 7. Plot training history

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("SMS Classifier Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 8. Implement the required prediction function

In [ ]:
def predict_message(message):
    """Return [spam_probability, predicted_label] for one SMS message.

    The first value is the model's estimated probability of spam (1).
    The second value is 'ham' or 'spam'.
    """
    message_vector = vectorizer.transform([str(message)]).astype("float32").toarray()
    spam_probability = float(model.predict(message_vector, verbose=0)[0][0])
    label = "spam" if spam_probability >= 0.5 else "ham"
    return [spam_probability, label]


## 9. Try example messages

In [ ]:
examples = [
    "Hey, are we still meeting for lunch today?",
    "Congratulations! You have won a free prize. Call now to claim your reward!",
    "I'll call you when I get home.",
    "URGENT! You have been selected for a cash prize. Text WIN to 80000."
]

for message in examples:
    print(message)
    print(predict_message(message))
    print()


## 10. Save the model and text vectorizer (optional)

The neural network and TF-IDF vectorizer are both needed for future predictions.


In [ ]:
import pickle

model.save("sms_spam_classifier.keras")
with open("sms_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved model and vectorizer.")
